In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

Adult Census Income Dataset Description

| Column | Description | Values / Type |
|--------|-------------|---------------|
| **age** | Age of individual (continuous) | 17-90 years |
| **workclass** | Employment type | Private, Self-emp-not-inc, Self-emp-inc, Federal-gov, Local-gov, State-gov, Without-pay, Never-worked |
| **fnlwgt** | Final weight - population weighting factor (number of people the census believes the entry represents) | Integer (large numbers) |
| **education** | Highest education level attained | Bachelors, HS-grad, 11th, Masters, Assoc-acdm, Assoc-voc, Doctorate, Prof-school, Some-college, 10th, 9th, 7th-8th, 5th-6th, 1st-4th, Preschool |
| **education-num** | Number of years of education | 1-16 years |
| **marital-status** | Marital status | Married-civ-spouse, Divorced, Never-married, Separated, Widowed, Married-spouse-absent, Married-AF-spouse |
| **occupation** | Job type | Adm-clerical, Exec-managerial, Handlers-cleaners, Prof-specialty, Other-service, Sales, Craft-repair, Transport-moving, Farming-fishing, Machine-op-inspct, Tech-support, Protective-serv, Armed-Forces, Priv-house-serv |
| **relationship** | Family relationship status | Wife, Husband, Own-child, Not-in-family, Other-relative, Unmarried |
| **race** | Race category | White, Black, Asian-Pac-Islander, Amer-Indian-Eskimo, Other |
| **sex** | Gender | Male, Female |
| **capital-gain** | Capital gains (income from investments) | 0 to large positive numbers |
| **capital-loss** | Capital losses (losses from investments) | 0 to large positive numbers |
| **hours-per-week** | Hours worked per week | 1-99 hours |
| **native-country** | Country of origin | United-States, Cuba, Mexico, Philippines, etc. |
| **income** | **Target variable** — Income level | <=50K, >50K |

In [23]:
column_names = [
    'age',
    'workclass',
    'fnlwgt',
    'education',
    'education_num',
    'marital_status',
    'occupation',
    'relationship',
    'race',
    'sex',
    'capital_gain',
    'capital_loss',
    'hours_per_week',
    'native_country',
    'income'
]

In [48]:
data = pd.read_csv('../datasets/adults.csv', names=column_names, skipinitialspace=True)
data.head()

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [25]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             32561 non-null  int64 
 1   workclass       32561 non-null  object
 2   fnlwgt          32561 non-null  int64 
 3   education       32561 non-null  object
 4   education_num   32561 non-null  int64 
 5   marital_status  32561 non-null  object
 6   occupation      32561 non-null  object
 7   relationship    32561 non-null  object
 8   race            32561 non-null  object
 9   sex             32561 non-null  object
 10  capital_gain    32561 non-null  int64 
 11  capital_loss    32561 non-null  int64 
 12  hours_per_week  32561 non-null  int64 
 13  native_country  32561 non-null  object
 14  income          32561 non-null  object
dtypes: int64(6), object(9)
memory usage: 3.7+ MB


### Create socioeconomic classes and analyze wealth distribution patterns.

Create Wealth Index

Combine the following features into a single `wealth_index`:

| Feature | Weight / Method |
|---------|-----------------|
| `capital-gain` | Normalized (0-1) |
| `capital-loss` | Normalized (0-1) |
| `education-num` | Normalized (0-1) |

**Formula:**

```python
wealth_index = (normalized_capital_gain + normalized_capital_loss + normalized_education_num) / 3
```

Stratify individuals into **4 socioeconomic classes** using quartiles of `wealth_index`:

| Class | Quartile Range |
|-------|----------------|
| **Lower** | Q1 (0-25%) |
| **Lower-Middle** | Q2 (25-50%) |
| **Upper-Middle** | Q3 (50-75%) |
| **Upper** | Q4 (75-100%) |



For each socioeconomic class, calculate the following metrics:

| Metric | Description |
|--------|-------------|
| **Mean hours-per-week** | Average hours worked per week |
| **Percentage with income >50K** | % of individuals earning above $50K |
| **Most common occupation** | Most frequent occupation in the class |
| **Education distribution** | Top 3 education levels in the class |

Find the socioeconomic class with the **highest representation** in the `>50K` income group.

In [49]:
scaler_cg = MinMaxScaler()
scaler_cl = MinMaxScaler()
scaler_en = MinMaxScaler()

cg_normalized = scaler_cg.fit_transform(data[['capital_gain']]).flatten()
cl_normalized = scaler_cl.fit_transform(data[['capital_loss']]).flatten()
en_normalized = scaler_en.fit_transform(data[['education_num']]).flatten()

data['wealth_index'] = (cg_normalized + cl_normalized + en_normalized) / 3

In [50]:
data[['education_num', 'capital_gain', 'capital_loss', 'wealth_index']].head()

,education_num,capital_gain,capital_loss,wealth_index
0,13,2174,0,0.273913
1,13,0,0,0.266667
2,9,0,0,0.177778
3,7,0,0,0.133333
4,13,0,0,0.266667


In [51]:
quartile = [0, .25, .5, .75, 1.]
labels = ['lower', 'lower-middle', 'upper-middle', 'upper']
data['wealth_class'] = pd.qcut(x=data['wealth_index'], q=[0, .25, .5, .75, 1.], labels=labels)
data['class_index'] = data['wealth_class'].map({
    'lower': 1,
    'lower-middle': 2,
    'upper-middle': 3,
    'upper': 4
})

In [52]:
data[['education_num', 'capital_gain', 'capital_loss', 'wealth_index', 'wealth_class', 'class_index']].head()

,education_num,capital_gain,capital_loss,wealth_index,wealth_class,class_index
0,13,2174,0,0.273913,upper,4
1,13,0,0,0.266667,upper-middle,3
2,9,0,0,0.177778,lower,1
3,7,0,0,0.133333,lower,1
4,13,0,0,0.266667,upper-middle,3


In [111]:
def wealth_stats(group):
    count_with_income_gt_50k = group[group['income']=='>50K'].shape[0]
    count_total = group.shape[0]
    
    mode_occupation = group['occupation'].mode()
    most_common_occupation = mode_occupation[0] if not mode_occupation.empty else np.nan

    education_counts = group['education'].value_counts()
    top_3_education = education_counts.head(3).index.tolist()
        
    pct_with_income_gt_50k = round((group['income']=='>50K').mean() * 100, 2)
    
    return pd.Series(
        {
            'mean_hours_per_weak': group['hours_per_week'].mean().round(2),
            'percentage_with_income_gt_50k': round(count_with_income_gt_50k * 100 /count_total, 2),
            'most_common_occupation': most_common_occupation,
            'top_3_education': top_3_education,
            'percentage_with_income_gt_50k_2': pct_with_income_gt_50k
        }
    )
    
data.groupby(['wealth_class'], observed=True).apply(wealth_stats, include_groups=False).reset_index()

,wealth_class,mean_hours_per_weak,percentage_with_income_gt_50k,most_common_occupation,top_3_education,percentage_with_income_gt_50k_2
0,lower,39.27,10.61,Craft-repair,"[HS-grad, 11th, 10th]",10.61
1,lower-middle,38.62,15.86,Adm-clerical,"[Some-college, HS-grad, 7th-8th]",15.86
2,upper-middle,41.81,33.17,Prof-specialty,"[Bachelors, Assoc-voc, Assoc-acdm]",33.17
3,upper,44.43,61.56,Prof-specialty,"[Masters, Bachelors, Prof-school]",61.56


In [89]:
occupation = data.groupby(['wealth_class', 'occupation'], observed=True).size().reset_index().rename(columns={0: 'count'})
occupation['rank'] = occupation.groupby(['wealth_class'], observed=True)['count'].rank(method='dense', ascending=False)
most_common_occupation = occupation[occupation['rank']==1]
most_common_occupation

,wealth_class,occupation,count,rank
3,lower,Craft-repair,2337,1.0
16,lower-middle,Adm-clerical,1225,1.0
40,upper-middle,Prof-specialty,1581,1.0
55,upper,Prof-specialty,1924,1.0


In [90]:
occupation = data.groupby(['wealth_class', 'education'], observed=True).size().reset_index().rename(columns={0: 'count'})
occupation['rank'] = occupation.groupby(['wealth_class'], observed=True)['count'].rank(method='dense', ascending=False)
occupation[occupation['rank']<=3]

,wealth_class,education,count,rank
0,lower,10th,902,3.0
1,lower,11th,1139,2.0
7,lower,HS-grad,9415,1.0
14,lower-middle,7th-8th,11,3.0
15,lower-middle,HS-grad,448,2.0
16,lower-middle,Some-college,6533,1.0
24,upper-middle,Assoc-acdm,972,3.0
25,upper-middle,Assoc-voc,1308,2.0
26,upper-middle,Bachelors,4384,1.0
36,upper,Bachelors,971,2.0


### Investigate the relationship between hours worked, education, and income to find anomalies.

Step 1: Create Hours Category

Create `hours_category` based on weekly hours:

| Category | Hours Range |
|----------|-------------|
| **Part-time** | < 30 hours |
| **Full-time** | 30 - 45 hours |
| **Overtime** | 46 - 60 hours |
| **Extreme** | > 60 hours |

Step 2: Calculate Metrics per Occupation

For each occupation, calculate the following:

| Metric | Description |
|--------|-------------|
| **Average hours-per-week** | Mean hours worked per week |
| **Average education-num** | Mean years of education |
| **Income >50K percentage** | % of individuals earning above $50K |

Step 3: Create Efficiency Score

$$
\text{efficiency\_score} = \frac{\text{income\_level}}{\text{hours\_per\_week}}
$$

Where:
- **income_level** = `1` if income >50K, else `0`

Step 4: Identify High & Low Efficiency Roles

| Category | Pattern |
|----------|---------|
| **High-Efficiency** | Low hours, high income |
| **Low-Efficiency** | High hours, low income |

Step 5: Flag High Efficiency Occupations

Flag occupations with **efficiency_score > 0.02** as `'High Efficiency'`.
